In [ ]:
import sys
print(sys.executable)

/mnt/c/users/omerb/OneDrive/omer/Projects/ai/.venv/bin/python


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

AttributeError: partially initialized module 'torch' from '/mnt/c/users/omerb/OneDrive/omer/Projects/ai/.venv/lib/python3.14/site-packages/torch/__init__.py' has no attribute 'library' (most likely due to a circular import)

In [ ]:
#-----------------------------------------------------------------------------------
class Embedding(nn.Module):
    """
    Embeds vocub in the linear space
    """
    def __init__(self, dvocub, dembedding):
        
        super().__init__()
        self.dvocub = dvocub
        self.dembedding = dembedding
        self.embed = nn.Linear(dvocub, dembedding)
    
    def forward(self,x):
        return self.embed[x]
    
#------------------------------------------------------------------------------------
class PositionalEmbedding(nn.Module):
    """
    Just holds the vector offset we add
        """
    def __init__(self, context_size, dembedding):
        super().__init__()
        self.offset = nn.Linear(context_size,dembedding)
    
    def forward(self,x):
        return self.offset
    

"""
As a prep for the transformer block, let's repeat how it works

dmodel = context*dembedding is the size that will be propagating through the model.

tokens->vocub->embeddings->embed+position->(attention_heads= xQ is the query matrix, xK is the key matrix. Their rows
are the questions\keys. Then (xQ)(xK)^t has (i,j) how xj answers xi. Then if j>i we mask, so we get upper triangular.
Then we concanate, then layer norm, then add to x. So

just does the heads part

Parameters-
context
dembedding 
dcorrelation
    Q,K will be (embed,correlation)
dhidden
    V will be (correlation, dhidden)
"""
#-------------------------------------------------------------------------------------
class AttentionHead(nn.Module):
    def __init__(self, dembedding, dcorrelation,dhidden):
        super().__init__()
        self.dembedding = dembedding
        self.dcorrelation = dcorrelation #this is the correlation_space
        self.dhidden = dhidden
        self.Q = nn.Linear(dembedding, dcorrelation)
        self.K = nn.Linear(dembedding, dcorrelation)
        self.V = nn.Linear(dembedding,dhidden)

    def forward(self,x):
        """
        x.shape = (batch,context, dembedding)
        """
        assert(x.shape[2] == self.dembedding)
        context = x.shape[1]
        xQ = x@self.Q
        xK = x@self.K
        xV = x@self.V
        #these have shape (batch, context, dcorrelation\dhidden)
        #b = batch, c,d = context, k = korrelation
        if self.training:
            xQxKt = torch.einsum("bck,bdk->bcd", xQ, xK)
        else:
            xQxKt = torch.einsum("ck,dk->cd", xQ,xK)
        nxQxKt = xQxKt / torch.sqrt(self.dcorrelation)
        upper_triangular = torch.tensor([[float("-inf") if i<j else 0 for j in range(context) ] for i in range(context)])
        if self.training:
            bupper_triangular = upper_triangular.unsqueeze(0)
            masked_xQxKt = nxQxKt + bupper_triangular #broadcast
        else:
            masked_xQxKt = masked_xQxKt + upper_triangular
        probs = masked_xQxKt.softmax(dim=-1)
        #shape is (Batch,context,context) with (i,j) is how j answers i
        if self.training:
            changes = torch.einsum("bcd,bdh->bch", probs, xV)
        else:
            changes = torch.einsum("cd,dh->ch", probs, xV)
        #if self.trianing: batch, context, hidden
        #if inference: context, hidden
        return changes
        
"""
Parameters-
context
heads
dembedding
dcorrelation
dhidden

x=x+(projection(heads(layer(x)))).
x = x+(ffn(layer(x))).
Here both layer, ffn are per token-vector
"""
#-------------------------------------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, context, dheads, dembedding, dcorrelation, dhidden):
        super().__init__()

        self.context = context
        self.dheads = dheads
        self.dembedding= dembedding
        self.dcorrelation = dcorrelation
        self.dhidden = dhidden
        self.dmodel = context*dembedding

        self.LN1 = nn.LayerNorm(dembedding)
        self.heads = nn.ModuleList()
        for i in range(dheads):
            head = AttentionHead(context,dembedding,dcorrelation, dhidden)
            self.heads.append(head)
        self.projection = nn.Linear(dhidden*dheads , self.dembedding)

        self.LN2 = nn.LayerNorm(dembedding)
        self.L2 = nn.Linear(dembedding, dembedding)
        self.TanH = nn.TanH(dembedding)
    
    def forward(self,x):
        #in training, x.shape = batch, context, dembed
        #in inference, x = context, dembed
        if self.training:
            batch = x.shape[0]
        
        LN1x = self.LN1(x,self.dembedding)
        heads_x = torch.stack([head(LN1x) for head in self.heads],dim=-1)
        #in training each of those head(LN1x) has shape batch,context,hidden,heads
        #in inference each of those head(LN1x) has shape context, hidden,heads
        if self.training:
            heads_x = heads_x.view(batch, self.context, self.dhidden*self.dheads)
        else:
            heads_x = heads_x.view(self.context, self.dhidden*self.dheads)
        x = x + heads_x@self.projection
        #in training has shape batch, context, dembedding
        #in inference has shape context, dembedding

        LN2x = self.LN2(x, self.dembedding)
        y = LN2x @ self.L2
        z = self.TanH(y)
        x = x + z
        #in training has shape batch, context, dembedding
        #in inference has shape context, deembedding
        return x
    
#------------------------------------------------------------------------------------
class Transformer(nn.Module):
    def __init__(self, dvocub, context, dheads, dembedding, dcorrelation, dhidden, blocks = 1):
        super().__init_()
        self.dvocub, self.context, self.dheads, self.dembedding, self.dcorrelation, self.dhidden, self.blocks = dvocub, context, dheads, dembedding, dcorrelation, dhidden, blocks

        self.embedding = Embedding(dvocub, dembedding)
        self.position_shift = PositionalEmbedding(context, dembedding)
        self.blocks = nn.ModuleLayer([TransformerBlock(context, dheads, dembedding, dcorrelation, dhidden) for i in range(blocks)])

    def forward(self, w):
        """
        x= embed(w)
        x = x + positional(x)
        for each block:
            x = block(x)
        return x
        """

<>:35: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.
<>:35: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.
/tmp/ipykernel_1088/4148768008.py:35: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.
  are the questions\keys. Then (xQ)(xK)^t has (i,j) how xj answers xi. Then if j>i we mask, so we get upper triangular.


NameError: name 'nn' is not defined